In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv("Ice Cream Sales and Temperature.csv")

In [5]:
df.head()

,Date,Time,Temperature (Celsius),Ice Cream Sales
0,2023-01-01,08:00,18,20
1,2023-01-01,08:05,19,22
2,2023-01-01,08:10,20,25
3,2023-01-01,08:15,21,24
4,2023-01-01,08:20,22,26


In [6]:
df.shape

(45, 4)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Date                   45 non-null     str  
 1   Time                   45 non-null     str  
 2   Temperature (Celsius)  45 non-null     int64
 3   Ice Cream Sales        45 non-null     int64
dtypes: int64(2), str(2)
memory usage: 2.2 KB


In [8]:
df.isnull().sum().sum()

np.int64(0)

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df = df.drop(columns=["Date", "Time"]).drop_duplicates().dropna()
df.columns = ["temperature", "sales"]

In [12]:
X = df[["temperature"]]
y = df["sales"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
def build_pipeline(degree=1):
    steps = []
    if degree > 1:
        steps.append(("poly", PolynomialFeatures(degree, include_bias=False)))
    steps += [("scaler", StandardScaler()),
              ("model", LinearRegression())]
    return Pipeline(steps)

In [17]:
results = []
for name, deg in [("Linear (deg 1)", 1), ("Polynomial (deg 2)", 2), ("Polynomial (deg 3)", 3)]:
    pipe = build_pipeline(deg)
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2").mean()
    pipe.fit(X_train, y_train)
    pred_tr, pred_te = pipe.predict(X_train), pipe.predict(X_test)
    results.append({
        "Model": name,
        "Train R2": r2_score(y_train, pred_tr),
        "CV R2 (5-fold)": cv,
        "Test R2": r2_score(y_test, pred_te),
        "Test MAE": mean_absolute_error(y_test, pred_te),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, pred_te)),
    })
res = pd.DataFrame(results).round(3)
print(res.to_string(index=False))

             Model  Train R2  CV R2 (5-fold)  Test R2  Test MAE  Test RMSE
    Linear (deg 1)     0.729           0.695    0.882     3.695      4.116
Polynomial (deg 2)     0.889           0.869    0.927     2.806      3.231
Polynomial (deg 3)     0.964           0.953    0.980     1.315      1.700
